# 03 · Exploratory analysis and storytelling
**Renewable Energy Transition in France: Regional Dynamics and the Rise of Sustainable Electricity Production**

Author: Soumahoro Souleymane · EFREI Paris · Data Storytelling project (#EFREIDataStories2026)
Research question: *How are French regions progressing toward a renewable electricity future?*

The data is ready (notebook 02). In this notebook I look for the story. I do not want a list of charts: I want to find the one message that the data supports best, and to know which numbers prove it.

**How I work**
- I go through the five levels of the brief: France overall, regions, sources, time, transition indicators.
- Each figure answers one business question and has a card with *Business question, Observation, Interpretation, Storytelling value*. The numbers in the observations are computed by the notebook, so they cannot differ from the figures.
- At the end I rank all the findings (Major, Supporting, Interesting), I write three possible stories and I choose one.
- I do not build the dashboard here. I only write down what it should contain.

**The eight questions of the brief**

| # | Question | Where |
|---|---|---|
| Q1 | How has renewable production evolved since 2014? | 2 |
| Q2 | Which regions lead the transition? | 4 |
| Q3 | Which regions fall behind? | 4 |
| Q4 | Which renewable source grows fastest? | 3 |
| Q5 | How has the renewable share evolved? | 2 |
| Q6 | Which seasonal patterns exist? | 5 |
| Q7 | Are the regional gaps increasing or decreasing? | 6 |
| Q8 | Which findings are surprising? | 7, 8 |

## 0 · Setup and loading
I only read the files produced by notebook 02 (`data/processed/`). Nothing is cleaned again here. Figures are saved in `reports/figures/` so that the reports and the app can reuse them.

In [ ]:
import sys, os, json, importlib, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy import stats
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "utils" / "prep.py").exists():
            return p
    raise FileNotFoundError("Folder utils/ not found. Put it next to notebooks/ and data/.")


ROOT = find_root()
sys.path.insert(0, str(ROOT))
from utils import config as cfg, viz
importlib.reload(cfg); importlib.reload(viz)

PROC, REPORTS = ROOT / "data" / "processed", ROOT / "reports"
FIGS = REPORTS / "figures"
FIGS.mkdir(parents=True, exist_ok=True)
viz.set_style()
cards = viz.Cards(FIGS)
print("project root:", ROOT)

In [ ]:
feat = pd.read_csv(PROC / "feature_engineered_dataset.csv", parse_dates=["date"], dtype={"region_code": str})
annual = pd.read_csv(PROC / "annual_region_summary.csv", dtype={"region_code": str})
nat = pd.read_csv(PROC / "national_annual.csv")
fact = pd.read_csv(PROC / "fact_energy.csv", parse_dates=["date"], dtype={"region_code": str})
dimr = pd.read_csv(PROC / "dim_region.csv", dtype={"region_code": str})
geo = json.loads((PROC / "regions.geojson").read_text(encoding="utf-8"))

print(f"feature dataset : {feat.shape}   annual : {annual.shape}   national : {nat.shape}   fact : {fact.shape}   regions : {dimr.shape}")
Y0, Y1 = int(annual.year.min()), int(annual.year.max())
YEARS = Y1 - Y0
print(f"period : {feat.date.min():%Y-%m} to {feat.date.max():%Y-%m}  ({Y0}-{Y1}, {YEARS} years between the first and the last year)")

HAS_NUC = dimr.set_index("region_code")["has_nuclear"].to_dict()
NAME = dimr.set_index("region_code")["region"].to_dict()
SHORT = dimr.set_index("region_code")["region_short"].to_dict()
src = cfg.SOURCES
gcols = [cfg.GWH[s] for s in src]

# the annual tables must match the monthly table (nothing was lost between notebooks)
chk = feat.groupby("year")[["total_gwh", "renewable_gwh"]].sum()
assert np.allclose(chk["renewable_gwh"].to_numpy(), nat.set_index("year")["renewable_gwh"].reindex(chk.index).to_numpy())
# fact_energy keeps the raw net values; the feature table has the negative cells set to 0 (notebook 02, section 7.0)
diff = fact["production_gwh"].sum() - feat[gcols].sum().sum()
assert np.isclose(fact["production_gwh"].clip(lower=0).sum(), feat[gcols].sum().sum())
print(f"fact vs feature totals: difference of {diff:.1f} GWh = the negative net values set to 0")
print("consistency checks between monthly, annual, national and fact tables: OK")

In [ ]:
# national monthly series (sum of the 13 regions, so the shares are weighted by production)
nat_m = feat.groupby("date")[gcols + ["total_gwh", "renewable_gwh", "new_renewables_gwh"]].sum()
for s in src:
    nat_m[f"{s}_share_pct"] = nat_m[cfg.GWH[s]] / nat_m["total_gwh"] * 100
nat_m["renewable_share_pct"] = nat_m["renewable_gwh"] / nat_m["total_gwh"] * 100
nat_m["new_renewables_share_pct"] = nat_m["new_renewables_gwh"] / nat_m["total_gwh"] * 100
nat_m["renewable_share_12m_pct"] = nat_m["renewable_gwh"].rolling(12).sum() / nat_m["total_gwh"].rolling(12).sum() * 100
nat_m["month"] = nat_m.index.month

natv = nat.set_index("year")
ann = annual.copy()
a0, a1 = ann[ann.year == Y0].set_index("region_code"), ann[ann.year == Y1].set_index("region_code")
print("national monthly series:", nat_m.shape)
natv[["total_gwh", "renewable_gwh", "new_renewables_gwh", "renewable_share_pct", "new_renewables_share_pct"]].round(2)

**What I see.** The four tables agree with each other and the period is the full 2014-2025 range. The national numbers in this notebook are built from the sum of the 13 regions, so every national share is weighted by production. When I average regional shares, I say it explicitly ("mean of the regions"), because the two numbers are not the same (see section 7).

## 1 · France overall
First level of the brief. I start with the headline numbers, then the evolution of renewable production (Q1) and of the renewable share (Q5).

In [ ]:
def tot(y, col):
    return float(natv.loc[y, col])

early = [Y0, Y0 + 1, Y0 + 2]
late = [Y1 - 2, Y1 - 1, Y1]
def share_over(years, num):
    return natv.loc[years, num].sum() / natv.loc[years, "total_gwh"].sum() * 100

ren0, ren1 = tot(Y0, "renewable_gwh"), tot(Y1, "renewable_gwh")
new0, new1 = tot(Y0, "new_renewables_gwh"), tot(Y1, "new_renewables_gwh")
hyd0, hyd1 = tot(Y0, "hydro_gwh"), tot(Y1, "hydro_gwh")
sh0, sh1 = tot(Y0, "renewable_share_pct"), tot(Y1, "renewable_share_pct")
nsh0, nsh1 = tot(Y0, "new_renewables_share_pct"), tot(Y1, "new_renewables_share_pct")
nuc0, nuc1 = tot(Y0, "nuclear_share_pct"), tot(Y1, "nuclear_share_pct")
the0, the1 = tot(Y0, "thermal_share_pct"), tot(Y1, "thermal_share_pct")
tg0, tg1 = tot(Y0, "total_gwh"), tot(Y1, "total_gwh")
sh_early, sh_late = share_over(early, "renewable_gwh"), share_over(late, "renewable_gwh")

tiles = [
    ("Renewable production", f"{ren1/1000:,.0f} TWh", f"{(ren1/ren0-1)*100:+.0f} % vs {Y0}", ren1 >= ren0),
    ("Renewable share", f"{sh1:.1f} %", f"{sh1-sh0:+.1f} pts vs {Y0}", sh1 >= sh0),
    ("Wind + solar + bioenergy share", f"{nsh1:.1f} %", f"{nsh1-nsh0:+.1f} pts vs {Y0}", nsh1 >= nsh0),
    ("Hydro, part of renewable GWh", f"{hyd1/ren1*100:.0f} %", f"{(hyd1/ren1-hyd0/ren0)*100:+.0f} pts vs {Y0}", hyd1/ren1 <= hyd0/ren0),
    ("Nuclear share", f"{nuc1:.1f} %", f"{nuc1-nuc0:+.1f} pts vs {Y0}", None),
    ("Thermal (fossil) share", f"{the1:.1f} %", f"{the1-the0:+.1f} pts vs {Y0}", the1 <= the0),
]
fig, axes = plt.subplots(2, 3, figsize=(12, 4.6))
for ax, (lab, val, delta, good) in zip(axes.ravel(), tiles):
    ax.axis("off"); ax.grid(False)
    ax.add_patch(Rectangle((0, 0), 1, 1, transform=ax.transAxes, facecolor="#F6F6F4", edgecolor="#DDDDDD"))
    ax.text(0.06, 0.78, lab, transform=ax.transAxes, fontsize=10, color=viz.MUTED)
    ax.text(0.06, 0.40, val, transform=ax.transAxes, fontsize=24, fontweight="bold", color=viz.INK)
    ax.text(0.06, 0.12, delta, transform=ax.transAxes, fontsize=10, color=viz.MUTED if good is None else ("#009E73" if good else "#D55E00"))
viz.suptitle(fig, f"France in {Y1} compared with {Y0}", "National figures, sum of the 13 regions (weighted by production)", y=1.06)
viz.source_note(fig, y=0.02)
plt.tight_layout(); cards.save(fig, "F01_kpi_overview"); plt.show()

cards.add("F01", "France overall", "KPI overview",
    f"Where does France stand in {Y1}, compared with {Y0}, on the main indicators of the transition?",
    f"Renewable production is {ren1/1000:,.0f} TWh in {Y1} against {ren0/1000:,.0f} TWh in {Y0} ({(ren1/ren0-1)*100:+.0f} %). The renewable share {viz.direction(sh1-sh0)} from {sh0:.1f} % to {sh1:.1f} % ({sh1-sh0:+.1f} pts) and the share of wind, solar and bioenergy from {nsh0:.1f} % to {nsh1:.1f} %. Hydro represents {hyd1/ren1*100:.0f} % of the renewable GWh in {Y1} ({hyd0/ren0*100:.0f} % in {Y0}). Nuclear is {nuc1:.1f} % and thermal {the1:.1f} % of the production. To avoid a single year effect I also compare the 3-year averages: {sh_early:.1f} % in {early[0]}-{early[-1]} and {sh_late:.1f} % in {late[0]}-{late[-1]}.",
    viz.pick(sh1 - sh0, [(8, "The renewable share clearly moved: this is a real structural change and not a one-year effect."), (3, "The share moves in the right direction but not dramatically; the story must be about the composition and the regions more than about the national total."), (0, "The share barely moved, so the interesting part is probably inside the mix and between regions.")], "The national share went down, which would be a surprising starting point for the story.") + f" The 3-year averages ({sh_late-sh_early:+.1f} pts) {'confirm' if np.sign(sh_late-sh_early)==np.sign(sh1-sh0) else 'do not confirm'} the change between the two end years.",
    "Opening slide of the story: four or five numbers that the audience remembers, with the comparison to the starting year. They will become the KPI row of the dashboard.")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
years = natv.index.to_numpy()
bottom = np.zeros(len(years))
for s in ["hydro", "bioenergy", "wind", "solar"]:
    v = natv[cfg.GWH[s]].to_numpy() / 1000
    ax.bar(years, v, bottom=bottom, color=cfg.COLOR[s], label=cfg.LABEL[s], width=0.75)
    bottom += v
ax.set_ylabel("Renewable production (TWh)")
ax.set_xticks(years); ax.set_xticklabels(years, rotation=0)
ax2 = ax.twinx()
ax2.plot(years, natv["renewable_share_pct"], color=viz.INK, marker="o", lw=2, label="Renewable share (right axis)")
ax2.set_ylabel("Share of total production (%)"); ax2.grid(False)
ax2.set_ylim(0, max(natv["renewable_share_pct"]) * 1.35)
ax2.spines["right"].set_visible(True)
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.1))
chg = (ren1 / ren0 - 1) * 100
viz.suptitle(fig, f"Renewable output {viz.direction(chg)} {abs(chg):.0f} % between {Y0} and {Y1}",
             "Annual renewable production by source, France (13 regions)", y=1.0)
plt.tight_layout(); cards.save(fig, "F02_renewable_production"); plt.show()

add_tot = ren1 - ren0
add_by = {s: tot(Y1, cfg.GWH[s]) - tot(Y0, cfg.GWH[s]) for s in cfg.RENEWABLE}
add_new = sum(add_by[s] for s in cfg.NEW_RENEWABLES)
sh_new_add = add_new / add_tot * 100 if add_tot else np.nan
ry = natv["renewable_gwh"]
yoy = ry.pct_change() * 100
cards.add("F02", "France overall", "Renewable production since 2014 (Q1)",
    f"How has renewable electricity production evolved since {Y0}, and which sources drive it?",
    f"Renewable production {viz.direction(chg)} from {ren0/1000:,.0f} TWh to {ren1/1000:,.0f} TWh ({chg:+.0f} %, {viz.cagr(ren0, ren1, YEARS):.1f} % per year). The best year is {ry.idxmax()} ({ry.max()/1000:,.0f} TWh) and the weakest {ry.idxmin()} ({ry.min()/1000:,.0f} TWh). The largest yearly drop is {yoy.min():.1f} % in {yoy.idxmin()} and the largest rise {yoy.max():+.1f} % in {yoy.idxmax()}. Between {Y0} and {Y1}, wind {add_by['wind']/1000:+.1f} TWh, solar {add_by['solar']/1000:+.1f} TWh, bioenergy {add_by['bioenergy']/1000:+.1f} TWh, hydro {add_by['hydro']/1000:+.1f} TWh.",
    f"Wind, solar and bioenergy add {add_new/1000:.1f} TWh, which is {sh_new_add:.0f} % of the net increase" + (f" (more than 100 % because hydro lost {abs(add_by['hydro'])/1000:.1f} TWh)" if sh_new_add > 100 else "") + (": hydro is almost stable and only follows the weather from one year to another." if abs(add_by['hydro']) < 0.25 * abs(add_tot) else ": hydro still plays a visible part in the change.") + f" The yearly series is not a straight line: its shape depends on the weather (hydro, wind) as much as on new installations.",
    "Shows that the transition is a story about new sources and not about the old hydro base. It is the first visual of the story and the easiest to read for a non-specialist.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), gridspec_kw={"width_ratios": [1, 1.25]})
ax = axes[0]
for col, lab, c, ls in [("renewable_share_pct", "Renewables", "#0072B2", "-"),
                        ("new_renewables_share_pct", "Wind + solar + bioenergy", "#D55E00", "-"),
                        ("hydro_share_pct", "Hydro", "#56B4E9", "--"),
                        ("thermal_share_pct", "Thermal", "#7F7F7F", "--")]:
    ax.plot(natv.index, natv[col], color=c, ls=ls, lw=2.2, marker="o", ms=3.5, label=lab)
    ax.text(natv.index[-1] + 0.15, natv[col].iloc[-1], f"{natv[col].iloc[-1]:.1f}", color=c, va="center", fontsize=9)
ax.set_xlim(Y0 - 0.3, Y1 + 1.2); ax.set_ylabel("% of national production"); ax.set_title("Annual shares")
ax.legend(loc="upper left", fontsize=8)
ax = axes[1]
ax.plot(nat_m.index, nat_m["renewable_share_pct"], color="#BBD7EC", lw=1, label="monthly")
ax.plot(nat_m.index, nat_m["renewable_share_12m_pct"], color="#0072B2", lw=2.4, label="rolling 12 months")
ax.set_ylabel("Renewable share (%)"); ax.set_title("Monthly and rolling 12-month renewable share")
ax.legend(loc="upper left")
viz.suptitle(fig, f"The renewable share {viz.direction(sh1-sh0)} from {sh0:.1f} % to {sh1:.1f} % in {YEARS} years", "Weighted national shares (sum of the 13 regions)", y=1.04)
plt.tight_layout(); cards.save(fig, "F03_renewable_share"); plt.show()

d_sh = natv["renewable_share_pct"].diff()
gain_total, gain_new = sh1 - sh0, nsh1 - nsh0
share_of_gain_new = gain_new / gain_total * 100 if gain_total else np.nan
cf = ren1 / tg0 * 100                                   # share if the total had stayed at its first-year level
den_effect = sh1 - cf
reg_mean1 = float(a1["renewable_share_pct"].mean())
cards.add("F03", "France overall", "Evolution of the renewable share (Q5)",
    f"How has the renewable share of electricity production evolved, and is the change steady?",
    f"The weighted national share {viz.direction(gain_total)} from {sh0:.1f} % to {sh1:.1f} % ({gain_total:+.1f} pts). Wind, solar and bioenergy alone go from {nsh0:.1f} % to {nsh1:.1f} % ({gain_new:+.1f} pts), which is {share_of_gain_new:.0f} % of the gain{' (more than 100 % because the hydro share fell)' if share_of_gain_new > 100 else ''}. The largest yearly step is {d_sh.max():+.1f} pts in {d_sh.idxmax()} and the largest fall {d_sh.min():+.1f} pts in {d_sh.idxmin()}. Total production {viz.direction(tg1-tg0, 'rose', 'fell')} by {abs(tg1/tg0-1)*100:.1f} % over the period: if it had stayed at the {Y0} level, the {Y1} renewable output would represent {cf:.1f} % instead of {sh1:.1f} % ({den_effect:+.1f} pts coming from the denominator).",
    f"The progress is not linear: some years go backwards, probably because of the weather (rain for hydro, wind). The rolling 12-month curve removes the seasonal wave and shows the underlying direction. " + ("A part of the gain comes from lower total production and not only from more renewables, so I must not present the whole gain as new capacity." if abs(den_effect) > 0.15 * abs(gain_total) else "The denominator effect is small, so the gain really comes from more renewable production."),
    "Answers the main question at national level and gives the first tension of the story (the trend is positive but irregular). It also gives a warning about how to read shares.")

## 2 · Renewable sources
Third level of the brief. Q4: which renewable source grows fastest? "Fastest" can mean two things, the growth rate and the absolute volume added, so I look at both. I also compare the first three and the last three years, because one year can be unusual (weather).

In [ ]:
idx = natv[gcols].div(natv.loc[Y0, gcols]) * 100
rows = []
for s in src:
    c = cfg.GWH[s]
    v0, v1 = natv.loc[Y0, c], natv.loc[Y1, c]
    e, l = natv.loc[early, c].mean(), natv.loc[late, c].mean()
    rows.append(dict(source=cfg.LABEL[s], key=s, gwh_start=v0, gwh_end=v1, added_twh=(v1 - v0) / 1000,
                     cagr_pct=viz.cagr(v0, v1, YEARS), cagr_3y_avg_pct=viz.cagr(e, l, (late[1] - early[1])),
                     multiple=v1 / v0 if v0 > 0 else np.nan, share_start=natv.loc[Y0, f"{s}_share_pct"], share_end=natv.loc[Y1, f"{s}_share_pct"]))
growth_src = pd.DataFrame(rows).set_index("source")
display(growth_src.drop(columns="key").round(2))

fig, axes = plt.subplots(1, 3, figsize=(14, 4.8), gridspec_kw={"width_ratios": [1.5, 1, 1]})
ax = axes[0]
for s in src:
    ax.plot(idx.index, idx[cfg.GWH[s]], color=cfg.COLOR[s], lw=2.2, label=cfg.LABEL[s])
    ax.text(Y1 + 0.2, idx[cfg.GWH[s]].iloc[-1], f"{cfg.LABEL[s]} {idx[cfg.GWH[s]].iloc[-1]:.0f}", color=cfg.COLOR[s], fontsize=8, va="center")
ax.set_yscale("log"); ax.set_xlim(Y0, Y1 + 2.8); ax.set_ylabel(f"Index ({Y0} = 100, log scale)"); ax.set_title("Production index")
order = growth_src.sort_values("cagr_pct")
ax = axes[1]
ax.barh(order.index, order["cagr_pct"], color=[cfg.COLOR[k] for k in order["key"]])
for i, v in enumerate(order["cagr_pct"]):
    ax.text(v + (0.3 if v >= 0 else -0.3), i, f"{v:.1f} %", va="center", ha="left" if v >= 0 else "right", fontsize=8)
ax.set_title("Growth rate per year"); ax.set_xlabel("% per year"); ax.axvline(0, color="#999", lw=0.8); ax.margins(x=0.28)
order2 = growth_src.sort_values("added_twh")
ax = axes[2]
ax.barh(order2.index, order2["added_twh"], color=[cfg.COLOR[k] for k in order2["key"]])
for i, v in enumerate(order2["added_twh"]):
    ax.text(v + (0.3 if v >= 0 else -0.3), i, f"{v:+.1f}", va="center", ha="left" if v >= 0 else "right", fontsize=8)
ax.set_title("Volume added"); ax.set_xlabel(f"TWh, {Y1} minus {Y0}"); ax.axvline(0, color="#999", lw=0.8); ax.margins(x=0.28)
ren_src = growth_src.loc[[cfg.LABEL[s] for s in cfg.RENEWABLE]]
fast, big = ren_src["cagr_pct"].idxmax(), ren_src["added_twh"].idxmax()
viz.suptitle(fig, f"{fast} grows fastest, {big} adds the most electricity", "Annual production by source, France", y=1.04)
plt.tight_layout(); cards.save(fig, "F04_source_growth"); plt.show()

fastest_row, biggest_row = ren_src.loc[fast], ren_src.loc[big]
slow = ren_src["cagr_pct"].idxmin()
cards.add("F04", "Sources", "Which renewable source grows fastest (Q4)",
    "Which renewable source grows fastest, and is it also the one that adds the most electricity?",
    f"{fast} has the highest growth rate ({fastest_row['cagr_pct']:.1f} % per year, x{fastest_row['multiple']:.1f}) and {big} the largest volume added ({biggest_row['added_twh']:+.1f} TWh). The ranking by growth rate with 3-year averages at both ends gives {ren_src['cagr_3y_avg_pct'].idxmax()} first ({ren_src['cagr_3y_avg_pct'].max():.1f} % per year). {slow} is the slowest renewable source ({ren_src.loc[slow, 'cagr_pct']:.1f} % per year, {ren_src.loc[slow, 'added_twh']:+.1f} TWh). Nuclear changes by {growth_src.loc['Nuclear', 'added_twh']:+.1f} TWh and thermal by {growth_src.loc['Thermal', 'added_twh']:+.1f} TWh.",
    ("The fastest source and the biggest source are not the same: a high rate on a small base (solar at the start) does not give the same volume as a moderate rate on a large base. " if fast != big else f"{fast} is both the fastest and the largest contributor, so the message is simple. ") + "I will always give the rate and the volume together on the dashboard, because one of them alone would mislead.",
    "Direct answer to Q4. The contrast rate vs volume is a good explanation moment for the audience.")

fig, (ax, axr) = plt.subplots(1, 2, figsize=(13.5, 4.6), gridspec_kw={"width_ratios": [1.1, 1]})
d = {cfg.LABEL[s]: natv.loc[Y1, f"{s}_share_pct"] - natv.loc[Y0, f"{s}_share_pct"] for s in src}
d = pd.Series(d).sort_values()
bars = ax.barh(d.index, d.values, color=[cfg.COLOR[[k for k, v in cfg.LABEL.items() if v == i][0]] for i in d.index])
for i, v in enumerate(d.values):
    ax.text(v + (0.15 if v >= 0 else -0.15), i, f"{v:+.1f} pts", va="center", ha="left" if v >= 0 else "right", fontsize=9)
ax.axvline(0, color="#777", lw=0.9); ax.set_xlabel(f"Change of the share in the national production, {Y0} to {Y1} (points)")
ax.set_xlim(min(d.min() * 1.35, -1), max(d.max() * 1.35, 1))
ax.set_title("Change of the share")
fam = pd.DataFrame({"Renewables": natv["renewable_gwh"], "Nuclear": natv["nuclear_gwh"], "Thermal": natv["thermal_gwh"]}) / 1000
for k_, c_ in [("Nuclear", cfg.COLOR["nuclear"]), ("Renewables", "#0072B2"), ("Thermal", cfg.COLOR["thermal"])]:
    axr.plot(fam.index, fam[k_], color=c_, lw=2.4, marker="o", ms=3.5, label=k_)
    axr.text(Y1 + 0.15, fam[k_].iloc[-1], f"{fam[k_].iloc[-1]:.0f}", color=c_, va="center", fontsize=9)
axr.set_xlim(Y0 - 0.3, Y1 + 1); axr.set_ylabel("TWh per year"); axr.set_title("Annual production by family"); axr.legend(fontsize=8, loc="center left")
nuc_low = int(fam["Nuclear"].idxmin())
up, dn = d.idxmax(), d.idxmin()
viz.suptitle(fig, f"Where does the change of the mix come from? {up} up, {dn} down", "Contribution of each source to the change of the national mix (sums to zero)", y=1.02)
plt.tight_layout(); cards.save(fig, "F05_mix_contribution"); plt.show()

pos_ren = sum(d[cfg.LABEL[s]] for s in cfg.RENEWABLE)
add_nuc, add_the = growth_src.loc["Nuclear", "added_twh"], growth_src.loc["Thermal", "added_twh"]
nuc_main = d["Nuclear"] < 0 and abs(d["Nuclear"]) > abs(d["Thermal"])
cards.add("F05", "Sources", "Contribution of each source to the change of the mix",
    "Which sources gained and which lost share in the national mix between the first and the last year, and what did the renewables replace?",
    f"The shares add up to 100 in both years, so the changes add up to zero. The renewables together gain {pos_ren:+.1f} pts: " + ", ".join(f"{cfg.LABEL[s]} {d[cfg.LABEL[s]]:+.1f}" for s in cfg.RENEWABLE) + f". Nuclear changes by {d['Nuclear']:+.1f} pts ({add_nuc:+.1f} TWh) and thermal by {d['Thermal']:+.1f} pts ({add_the:+.1f} TWh). The biggest gain is {up} ({d.max():+.1f} pts), the biggest loss {dn} ({d.min():+.1f} pts). Nuclear production is lowest in {nuc_low} ({fam['Nuclear'].min():.0f} TWh) and it moves by {fam['Nuclear'].iloc[-1]-fam['Nuclear'].min():+.0f} TWh between that year and {Y1}.",
    ("The renewables mostly take the place of nuclear and not of thermal: the thermal base was already small. " if nuc_main else "The renewables mostly take the place of thermal. " if d['Thermal'] < 0 else "The renewables grow next to the other sources. ") + f"Hydro moves by only {d['Hydro']:+.1f} pts, so the gain of the renewables is made of new sources. " + (f"Nuclear is not a straight line either (low point in {nuc_low}), so the size of the change depends on the two end years; the data does not say why nuclear output moved, I only describe it." if nuc_low not in (Y0, Y1) else "") ,
    "This is the 'what replaced what' chart. It gives an honest answer to the audience question: is the transition replacing fossil fuel, nuclear, or just growing next to them? The answer is not the one people expect, which makes it a strong point for the story.")

In [ ]:
mix = ann[ann.year.isin([Y0, Y1])].copy()
order_codes = a1.sort_values("renewable_share_pct").index.tolist()
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), sharey=True)
for ax, y in zip(axes, [Y0, Y1]):
    t = mix[mix.year == y].set_index("region_code").loc[order_codes]
    left = np.zeros(len(t))
    for s in ["thermal", "nuclear", "hydro", "bioenergy", "wind", "solar"]:
        v = t[f"{s}_share_pct"].to_numpy()
        ax.barh([SHORT[c] for c in t.index], v, left=left, color=cfg.COLOR[s], label=cfg.LABEL[s], height=0.75)
        left += v
    ax.set_xlim(0, 100); ax.set_title(str(y)); ax.set_xlabel("% of regional production")
    ax.grid(False)
axes[1].legend(ncol=6, loc="upper center", bbox_to_anchor=(-0.05, -0.1))
best, worst = a1["renewable_share_pct"].idxmax(), a1["renewable_share_pct"].idxmin()
viz.suptitle(fig, f"Regional energy mix: from {a1.loc[worst, 'renewable_share_pct']:.0f} % to {a1.loc[best, 'renewable_share_pct']:.0f} % renewable in {Y1}", f"Share of each source in the regional production, {Y0} and {Y1}, regions sorted by renewable share in {Y1}", y=1.03)
plt.tight_layout(); cards.save(fig, "F06_regional_mix"); plt.show()

nuc_regs = [c for c in a1.index if HAS_NUC[c]]
no_nuc_regs = [c for c in a1.index if not HAS_NUC[c]]
dom = pd.Series({c: cfg.LABEL[max(src, key=lambda s: a1.loc[c, f"{s}_share_pct"])] for c in a1.index}).value_counts()
dom_txt = ", ".join(f"{k} in {v}" for k, v in dom.items())
cards.add("F06", "Sources", "Energy mix by region",
    f"How different are the regional electricity mixes, and what is the main source of each region in {Y1}?",
    f"In {Y1} the renewable share goes from {a1.loc[worst, 'renewable_share_pct']:.1f} % ({NAME[worst]}) to {a1.loc[best, 'renewable_share_pct']:.1f} % ({NAME[best]}). The mean renewable share is {a1.loc[no_nuc_regs, 'renewable_share_pct'].mean():.1f} % in the {len(no_nuc_regs)} regions without nuclear and {a1.loc[nuc_regs, 'renewable_share_pct'].mean():.1f} % in the {len(nuc_regs)} regions with nuclear. The mean nuclear share of the second group is {a1.loc[nuc_regs, 'nuclear_share_pct'].mean():.0f} %. The main source of the region is: {dom_txt} (number of regions).",
    "There are two groups of regions with very different starting points. A region without nuclear plant can reach a high renewable share by producing little else, while a nuclear region has a large 'competitor' in its mix. Comparing the level of the share alone would therefore be unfair, and the progress is the better measure of effort.",
    "Sets the key tension of the regional story: not all regions start from the same place. It justifies using two measures (level and progress) in the rest of the project.")

## 3 · Regions
Second level of the brief, and the heart of the research question. Q2 and Q3 ask who leads and who falls behind. After figure F06 I know that the level of the renewable share mixes two things (the effort of the region and the absence of nuclear), so I always look at two measures:
- **level**: renewable share of the regional production in the last year,
- **progress**: change of this share since the first year, in percentage points.

In [ ]:
reg = pd.DataFrame({
    "region": pd.Series(NAME), "short": pd.Series(SHORT), "has_nuclear": pd.Series(HAS_NUC),
    "share0": a0["renewable_share_pct"], "share1": a1["renewable_share_pct"],
    "new0": a0["new_renewables_share_pct"], "new1": a1["new_renewables_share_pct"],
    "total0": a0["total_gwh"], "total1": a1["total_gwh"], "ren0": a0["renewable_gwh"], "ren1": a1["renewable_gwh"],
    "thermal1": a1["thermal_share_pct"], "nuclear1": a1["nuclear_share_pct"],
    "score1": a1["energy_transition_score"], "momentum1": a1["renewable_momentum_score"],
    "diversity1": a1["mix_diversity_12m"],
}).loc[a1.index]
reg["gain"] = reg["share1"] - reg["share0"]
reg["new_gain"] = reg["new1"] - reg["new0"]
reg["ren_added_twh"] = (reg["ren1"] - reg["ren0"]) / 1000
reg["rank_level"] = reg["share1"].rank(ascending=False, method="min").astype(int)
reg["rank_gain"] = reg["gain"].rank(ascending=False, method="min").astype(int)
reg["rank_level0"] = reg["share0"].rank(ascending=False, method="min").astype(int)
reg["rank_peer"] = reg.groupby("has_nuclear")["share1"].rank(ascending=False, method="min").astype(int)
med_level, med_gain = reg["share1"].median(), reg["gain"].median()
def quadrant(r):
    hi_l, hi_g = r["share1"] >= med_level, r["gain"] >= med_gain
    return "Leader" if hi_l and hi_g else "Catching up" if (not hi_l) and hi_g else "Resting on its level" if hi_l else "Lagging"
reg["profile"] = reg.apply(quadrant, axis=1)
display(reg.sort_values("share1", ascending=False)[["region", "has_nuclear", "share0", "share1", "gain", "new_gain", "rank_level", "rank_gain", "rank_peer", "profile"]].round(1))

In [ ]:
o = reg.sort_values("share1")
fig, ax = plt.subplots(figsize=(10.5, 6))
for i, (c, r) in enumerate(o.iterrows()):
    ax.plot([r["share0"], r["share1"]], [i, i], color="#C9C9C9", lw=2.5, zorder=1)
    ax.scatter(r["share0"], i, s=45, facecolor="white", edgecolor=viz.GROUP_COLOR[r["has_nuclear"]], lw=1.8, zorder=2)
    ax.scatter(r["share1"], i, s=70, color=viz.GROUP_COLOR[r["has_nuclear"]], zorder=3)
    ax.text(max(r["share0"], r["share1"]) + 1.5, i, f"{r['share1']:.0f} %  ({r['gain']:+.0f} pts)", va="center", fontsize=8.5, color=viz.INK)
ax.set_yticks(range(len(o))); ax.set_yticklabels(o["region"]); ax.set_xlabel("Renewable share of the regional production (%)")
ax.set_xlim(0, max(o["share1"]) * 1.2)
from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([], [], marker="o", color="w", markerfacecolor="white", markeredgecolor="#555", label=f"{Y0}", ms=7),
                   Line2D([], [], marker="o", color="w", markerfacecolor="#555", label=f"{Y1}", ms=8),
                   Line2D([], [], marker="s", color="w", markerfacecolor=viz.GROUP_COLOR[False], label="without nuclear", ms=8),
                   Line2D([], [], marker="s", color="w", markerfacecolor=viz.GROUP_COLOR[True], label="with nuclear", ms=8)],
          loc="lower right", ncol=2)
lead3 = ", ".join(reg.sort_values("share1", ascending=False)["region"].head(3))
viz.suptitle(fig, f"Level of the renewable share by region: {lead3} lead in {Y1}", f"Annual renewable share, {Y0} (empty) and {Y1} (full)", y=1.02)
plt.tight_layout(); cards.save(fig, "F07_regional_level"); plt.show()

top = reg.sort_values("share1", ascending=False)
bot = reg.sort_values("share1")
cards.add("F07", "Regions", "Regional ranking by level (Q2, Q3)",
    f"Which regions have the highest and the lowest renewable share in {Y1}, and how much did each move since {Y0}?",
    f"The three highest shares are {', '.join(f'{r.region} ({r.share1:.0f} %)' for r in top.head(3).itertuples())}. The three lowest are {', '.join(f'{r.region} ({r.share1:.0f} %)' for r in bot.head(3).itertuples())}. {int((reg['has_nuclear']==False).sum())} regions have no nuclear plant: {int((top.head(5)['has_nuclear']==False).sum())} of the 5 highest shares belong to them. The share rose in {int((reg['gain']>0).sum())} regions out of 13; the biggest rise is {reg.loc[reg['gain'].idxmax(), 'region']} ({reg['gain'].max():+.1f} pts) and the weakest change is {reg.loc[reg['gain'].idxmin(), 'region']} ({reg['gain'].min():+.1f} pts).",
    "The ranking by level answers 'who is the most renewable today' but it partly measures the absence of nuclear. For this reason I do not call the top regions 'leaders of the transition' yet: I need the progress too (next figure).",
    "Gives the audience a clear first answer to Q2 and Q3, and sets up a surprise: the level ranking is not the whole story.")

fig, ax = plt.subplots(figsize=(10.5, 6))
o = reg.sort_values("gain")
y = np.arange(len(o)); h = 0.38
ax.barh(y + h / 2, o["gain"], height=h, color="#0072B2", label="Renewables (all)")
ax.barh(y - h / 2, o["new_gain"], height=h, color="#D55E00", label="Wind + solar + bioenergy")
for i, (c, r) in enumerate(o.iterrows()):
    ax.text(r["gain"] + (0.4 if r["gain"] >= 0 else -0.4), i + h / 2, f"{r['gain']:+.1f}", va="center", ha="left" if r["gain"] >= 0 else "right", fontsize=8)
ax.set_yticks(y); ax.set_yticklabels(o["region"]); ax.axvline(0, color="#777", lw=0.9)
ax.set_xlabel(f"Change of the share, {Y0} to {Y1} (percentage points)"); ax.legend(loc="lower right")
prog3 = ", ".join(reg.sort_values("gain", ascending=False)["region"].head(3))
viz.suptitle(fig, f"Progress since {Y0}: {prog3} moved the most", "Change of the share of the regional production (points)", y=1.02)
plt.tight_layout(); cards.save(fig, "F08_regional_progress"); plt.show()

gp = reg.sort_values("gain", ascending=False)
overlap = len(set(gp.head(5).index) & set(top.head(5).index))
cards.add("F08", "Regions", "Regional ranking by progress (Q2, Q3)",
    f"Which regions progressed the most since {Y0}, and is progress the same as level?",
    f"The three biggest gains are {', '.join(f'{r.region} ({r.gain:+.1f} pts)' for r in gp.head(3).itertuples())}; the three smallest are {', '.join(f'{r.region} ({r.gain:+.1f} pts)' for r in gp.tail(3).itertuples())}. Only {overlap} region(s) are in both top 5 lists (level and progress). In TWh of renewable production, the three regions with the biggest gains add {', '.join(f'{v:+.1f}' for v in gp.head(3)['ren_added_twh'])} TWh. The gain of wind + solar + bioenergy is {reg['new_gain'].mean():.1f} pts on average, against {reg['gain'].mean():.1f} pts for all renewables: " + ("the new sources explain most of the gain." if reg['new_gain'].mean() > 0.6 * reg['gain'].mean() else "hydro and weather also move the regional shares."),
    ("The two rankings are different, which confirms that level and progress measure two different things. " if overlap <= 2 else "The two rankings overlap only partly, so level and progress measure related but different things. " if overlap == 3 else "The two rankings are close, so level and progress tell nearly the same story. ") + "A big gain in points can come from a small production (a small region or a small base), so I check the volume in TWh before calling a region a leader. " + f"The regions with the biggest gains are the ones to watch for the narrative of a transition 'in motion', whatever their starting point.",
    "This is the fairest answer to the research question (progress toward a renewable future) and it gives the regions names the audience can follow.")

In [ ]:
fig, ax = plt.subplots(figsize=(9.8, 6.4))
size = reg["total1"] / reg["total1"].max() * 900 + 60
for c, r in reg.iterrows():
    ax.scatter(r["share1"], r["gain"], s=size[c], color=viz.GROUP_COLOR[r["has_nuclear"]], alpha=0.85, edgecolor="white", lw=1)
    ax.annotate(r["short"], (r["share1"], r["gain"]), xytext=(6, 6), textcoords="offset points", fontsize=8.5, color=viz.INK)
ax.axvline(med_level, color="#888", ls="--", lw=1); ax.axhline(med_gain, color="#888", ls="--", lw=1)
xl, yl = ax.get_xlim(), ax.get_ylim()
ax.text(xl[1], yl[1], "Leaders ", ha="right", va="top", color=viz.MUTED, fontsize=10, fontweight="bold")
ax.text(xl[0], yl[1], " Catching up", ha="left", va="top", color=viz.MUTED, fontsize=10, fontweight="bold")
ax.text(xl[1], yl[0], "Resting on its level ", ha="right", va="bottom", color=viz.MUTED, fontsize=10, fontweight="bold")
ax.text(xl[0], yl[0], " Lagging", ha="left", va="bottom", color=viz.MUTED, fontsize=10, fontweight="bold")
ax.set_xlabel(f"Level: renewable share in {Y1} (%)"); ax.set_ylabel(f"Progress: change since {Y0} (points)")
ax.legend(handles=[Line2D([], [], marker="o", color="w", markerfacecolor=viz.GROUP_COLOR[k], label=viz.GROUP_LABEL[k], ms=9) for k in (False, True)], loc="upper center")
prof = reg.groupby("profile")["short"].apply(lambda s: ", ".join(s)).to_dict()
viz.suptitle(fig, "Level and progress give four different profiles of regions", f"Dashed lines = median of the 13 regions. Bubble size = total production {Y1}", y=1.02)
plt.tight_layout(); cards.save(fig, "F09_level_vs_progress"); plt.show()

sl = stats.linregress(reg["share0"], reg["gain"])
rho_lg = stats.spearmanr(reg["share0"], reg["gain"])
rho_rank = stats.spearmanr(reg["share0"], reg["share1"])
cards.add("F09", "Regions", "Level versus progress: four profiles of regions (Q2, Q3)",
    "Are the regions that start high the ones that progress most, and which regions can honestly be called leaders or laggards?",
    "Profiles (with the median as cut-off): " + "; ".join(f"{k}: {v}" for k, v in prof.items()) + f". Across regions, the correlation between the starting share ({Y0}) and the gain is {rho_lg.statistic:+.2f} (Spearman; slope {sl.slope:+.2f} pts per point of starting share, p = {sl.pvalue:.2f}, n = 13). The ranking of the regions is {'very stable' if rho_rank.statistic > 0.9 else 'fairly stable' if rho_rank.statistic > 0.7 else 'not stable'} between {Y0} and {Y1} (Spearman {rho_rank.statistic:.2f}).",
    ("There is no clear link between the starting level and the gain, so catching up is not systematic. " if abs(rho_lg.statistic) < 0.3 or sl.pvalue > 0.1 else "The regions that started low are catching up. " if rho_lg.statistic < 0 else "The regions that started high also progressed most, so the gaps grow. ") + f"With only 13 regions, the correlation is a clue and not a proof (p-value {sl.pvalue:.2f}).",
    "This chart is the core of the regional story: it replaces the single ranking with four profiles. 'Catching up' and 'Resting on its level' are the interesting groups for the narrative.")

In [ ]:
centroid = dimr.set_index("region_code")[["lon", "lat"]]
fig, axes = plt.subplots(1, 2, figsize=(11.5, 6))
vals1 = reg["share1"].to_dict()
labs1 = {c: (centroid.loc[c, "lon"], centroid.loc[c, "lat"], f"{reg.loc[c, 'short']}\n{reg.loc[c, 'share1']:.0f}") for c in reg.index}
viz.draw_map(axes[0], geo, vals1, cmap="YlGn", labels=labs1, title=f"Renewable share in {Y1} (%)", fig=fig, cbar_label="%")
gains = reg["gain"].to_dict()
labs2 = {c: (centroid.loc[c, "lon"], centroid.loc[c, "lat"], f"{reg.loc[c, 'short']}\n{reg.loc[c, 'gain']:+.0f}") for c in reg.index}
vmax = max(abs(reg["gain"].min()), abs(reg["gain"].max()))
viz.draw_map(axes[1], geo, gains, cmap="PuOr_r", vmin=-vmax, vmax=vmax, labels=labs2, title=f"Change since {Y0} (points)", fig=fig, cbar_label="points")
viz.suptitle(fig, "Where the renewable electricity is, and where it is moving", "Annual renewable share of the regional production", y=1.0)
plt.tight_layout(); cards.save(fig, "F10_maps"); plt.show()

geo_pairs = reg.sort_values("share1", ascending=False)
cards.add("F10", "Regions", "Maps of level and progress",
    "Is there a geographic pattern in the level and in the progress of the renewable share?",
    f"The highest levels are in {', '.join(geo_pairs.head(3)['region'])} and the lowest in {', '.join(geo_pairs.tail(3)['region'])}. The strongest gains are in {', '.join(gp.head(3)['region'])}. The mean gain is {reg.loc[reg['has_nuclear'], 'gain'].mean():.1f} pts in regions with nuclear and {reg.loc[~reg['has_nuclear'], 'gain'].mean():.1f} pts in regions without.",
    "I read the map as an orientation tool. The first map is dominated by the nuclear / no nuclear divide, the second map is the one that shows where the movement is. I need to say which map the audience is looking at, because they tell different stories.",
    "The map is the most immediate way for a general audience to see the regional story, and it will be the main page of the dashboard.")

fig, ax = plt.subplots(figsize=(11.5, 5.8))
hm = ann.pivot(index="region", columns="year", values="renewable_share_pct")
hm = hm.loc[reg.sort_values("share1", ascending=False)["region"]]
im = ax.imshow(hm.values, aspect="auto", cmap="YlGn")
ax.set_xticks(range(len(hm.columns))); ax.set_xticklabels(hm.columns); ax.set_yticks(range(len(hm))); ax.set_yticklabels(hm.index)
for i in range(hm.shape[0]):
    for j in range(hm.shape[1]):
        v = hm.values[i, j]
        ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=7.5, color="white" if v > np.nanmax(hm.values) * 0.6 else viz.INK)
ax.grid(False); [ax.spines[s].set_visible(False) for s in ax.spines]
cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02); cb.outline.set_visible(False); cb.set_label("%")
viz.suptitle(fig, "Renewable share by region and year", "Annual renewable share (%), regions sorted by the share in the last year", y=1.02)
plt.tight_layout(); cards.save(fig, "F11_heatmap_region_year"); plt.show()

rk = ann.pivot(index="region", columns="year", values="renewable_share_pct").rank(ascending=False, method="min")
moves = (rk[Y0] - rk[Y1]).sort_values(ascending=False)
best_year = hm.idxmax(axis=1)
cards.add("F11", "Regions", "Heatmap region x year",
    "When did each region move, and is the progress regular or made of jumps?",
    f"The biggest climb in rank between {Y0} and {Y1} is {moves.index[0]} ({moves.iloc[0]:+.0f} places) and the biggest fall is {moves.index[-1]} ({moves.iloc[-1]:+.0f} places). {int((moves == 0).sum())} region(s) keep exactly the same rank. The share of the leading region is {hm.iloc[0].iloc[-1]:.0f} % in {Y1}. The year of highest share is {int(best_year.mode().iloc[0])} for most regions (it is the last year for {int((best_year == Y1).sum())} of 13).",
    "Where the colour changes suddenly from one year to the next, a new installation or a hydro year is the likely cause; the data does not say which one, so I will check it with the national picture before telling a story about a region.",
    "Useful for the 'time' dimension of the regional story and as a drill-down view in the dashboard.")

## 4 · Time: seasonal patterns (Q6)
Renewable sources do not produce at the same moment of the year. Solar follows the sun, wind and hydro are stronger in winter and spring. This matters for the story, because a national share is an average over the year but electricity is used month by month.

In [ ]:
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
nm = nat_m.groupby("month")[gcols].mean() / 1000                      # TWh per month, average over the years
share_m = nat_m.groupby("month")["renewable_share_pct"].mean()
fig, axes = plt.subplots(1, 2, figsize=(13, 4.9))
ax = axes[0]
bottom = np.zeros(12)
for s in ["hydro", "bioenergy", "wind", "solar"]:
    ax.bar(MONTHS, nm[cfg.GWH[s]], bottom=bottom, color=cfg.COLOR[s], label=cfg.LABEL[s], width=0.78)
    bottom += nm[cfg.GWH[s]].to_numpy()
ax.set_ylabel("TWh per month (average of the years)"); ax.set_title("Renewable production by month"); ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.08))
ax = axes[1]
data = [feat.loc[feat.month == m, "renewable_share_pct"].dropna().to_numpy() for m in range(1, 13)]
bp = ax.boxplot(data, patch_artist=True, showfliers=True, flierprops=dict(markersize=3, alpha=0.4), medianprops=dict(color=viz.INK))
season_color = {12: "#56B4E9", 1: "#56B4E9", 2: "#56B4E9", 3: "#009E73", 4: "#009E73", 5: "#009E73", 6: "#E69F00", 7: "#E69F00", 8: "#E69F00", 9: "#D55E00", 10: "#D55E00", 11: "#D55E00"}
ax.set_xticks(range(1, 13)); ax.set_xticklabels(MONTHS)
for patch, m in zip(bp["boxes"], range(1, 13)):
    patch.set_facecolor(season_color[m]); patch.set_alpha(0.55)
ax.set_ylabel("Renewable share (%), region-months"); ax.set_title("Spread of the renewable share across regions and years")
pk, tr = share_m.idxmax(), share_m.idxmin()
viz.suptitle(fig, f"Renewable production peaks in {MONTHS[pk-1]} and is lowest in {MONTHS[tr-1]}", "Average over 2014-2025; boxplot = 13 regions x 12 years per month (blue winter, green spring, yellow summer, orange autumn)", y=1.04)
plt.tight_layout(); cards.save(fig, "F12_seasonal_profile"); plt.show()

peak = {s: int(nm[cfg.GWH[s]].idxmax()) for s in cfg.RENEWABLE}
trough = {s: int(nm[cfg.GWH[s]].idxmin()) for s in cfg.RENEWABLE}
amp = {s: (nm[cfg.GWH[s]].max() / nm[cfg.GWH[s]].min() if nm[cfg.GWH[s]].min() > 0 else np.nan) for s in cfg.RENEWABLE}
seas_sh = nat_m.groupby(nat_m.index.month.map(cfg.SEASON))[["renewable_gwh", "total_gwh"]].sum()
seas_sh = (seas_sh["renewable_gwh"] / seas_sh["total_gwh"] * 100)
tot_m = nat_m.groupby("month")["total_gwh"].mean(); nuc_m = nat_m.groupby("month")["nuclear_gwh"].mean()
cards.add("F12", "Time", "Seasonal profile (Q6)",
    "Which seasonal patterns exist in renewable production and in the renewable share?",
    f"The national renewable share is highest in {MONTHS[pk-1]} ({share_m.max():.1f} %) and lowest in {MONTHS[tr-1]} ({share_m.min():.1f} %); by season: " + ", ".join(f"{k} {v:.1f} %" for k, v in seas_sh.items()) + ". The production peaks in " + ", ".join(f"{cfg.LABEL[s].lower()} {MONTHS[peak[s]-1]}" for s in cfg.RENEWABLE) + f". The peak month is {amp['solar']:.1f} times the weakest month for solar, {amp['wind']:.1f} for wind and {amp['hydro']:.1f} for hydro. The boxes show that regions differ a lot inside the same month (median {np.median(data[0]):.0f} % in January).",
    "Solar and wind do not peak in the same month, so they fill each other's low periods only if the peaks really are in different seasons" + (": here they are." if abs(peak['solar'] - peak['wind']) >= 3 and abs(peak['solar'] - peak['wind']) <= 9 else ": here the peaks are close, so the complement is weak.") + " The renewable share that I quote for a year hides these swings, which matters for a story about reliability. " + f"The share also depends on the total: total production is lowest in {MONTHS[int(tot_m.idxmin())-1]} and nuclear production is lowest in {MONTHS[int(nuc_m.idxmin())-1]}, so a low nuclear month can raise the renewable share without any renewable growth.",
    "Brings a new dimension to the story: the transition is not only 'how much' but 'when'. It is also a good way to talk to a non-expert audience (sun in summer, wind in winter).")

In [ ]:
dec = viz.decompose(nat_m["renewable_gwh"] / 1000, 12)
fs, ft = viz.strengths(dec)
fig, axes = plt.subplots(4, 1, figsize=(11, 8.6), sharex=True)
for ax, (col, lab, c) in zip(axes, [("observed", "Observed (TWh)", "#0072B2"), ("trend", "Trend", "#D55E00"), ("seasonal", "Seasonal", "#009E73"), ("resid", "Residual", "#7F7F7F")]):
    ax.plot(dec.index, dec[col], color=c, lw=1.6)
    ax.set_ylabel(lab, fontsize=9)
    if col == "resid":
        ax.axhline(0, color="#999", lw=0.8)
viz.suptitle(fig, "Seasonal decomposition of the national renewable production", f"Additive decomposition, 12-month period. Seasonal strength {fs:.2f}, trend strength {ft:.2f} (0 to 1)", y=0.995)
plt.tight_layout(); cards.save(fig, "F13_decomposition"); plt.show()

tr_gain = dec["trend"].dropna().iloc[-1] - dec["trend"].dropna().iloc[0]
tr_first, tr_last = dec["trend"].dropna().iloc[0], dec["trend"].dropna().iloc[-1]
seas_range = dec["seasonal"].max() - dec["seasonal"].min()
res_max = dec["resid"].abs().idxmax()
cards.add("F13", "Time", "Seasonal decomposition of renewable production",
    "How much of the monthly variation of renewable production is season, how much is trend, and how much is something else (weather, accidents)?",
    f"The seasonal component moves between {dec['seasonal'].min():+.1f} and {dec['seasonal'].max():+.1f} TWh around the trend (range {seas_range:.1f} TWh). The trend {viz.direction(tr_gain)} from {tr_first:.1f} to {tr_last:.1f} TWh per month ({(tr_last/tr_first-1)*100:+.0f} %). The seasonal strength is {fs:.2f} and the trend strength {ft:.2f}. The largest residual is in {res_max:%Y-%m} ({dec.loc[res_max, 'resid']:+.1f} TWh).",
    (f"The season is the strongest pattern of the series ({fs:.2f}), stronger than the trend ({ft:.2f}). " if fs > ft else f"The trend ({ft:.2f}) is stronger than the season ({fs:.2f}) in this series. ") + "The residual contains what neither the season nor the trend explains, mainly the weather of the month; it is small compared to the seasonal wave, except in a few months that I can note for the story.",
    "Shows the difference between a trend that will keep growing and a seasonal wave that always comes back. It supports the message that the long-term direction is clear while the monthly values are very seasonal.")

comp = {}
for s in cfg.RENEWABLE:
    d_ = viz.decompose(nat_m[cfg.GWH[s]] / 1000, 12)
    comp[s] = pd.Series(d_["seasonal"].groupby(d_.index.month).mean()).to_numpy()
    comp[s + "_fs"], comp[s + "_ft"] = viz.strengths(d_)
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6), gridspec_kw={"width_ratios": [1.5, 1]})
ax = axes[0]
for s in cfg.RENEWABLE:
    ax.plot(MONTHS, comp[s], color=cfg.COLOR[s], lw=2.4, marker="o", ms=4, label=cfg.LABEL[s])
ax.axhline(0, color="#999", lw=0.8); ax.set_ylabel("Seasonal component (TWh)"); ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.08))
ax.set_title("Seasonal shape of each source")
ws = pd.DataFrame({"wind": comp["wind"], "solar": comp["solar"]})
ws["wind + solar"] = ws.sum(axis=1)
rho_ws = np.corrcoef(comp["wind"], comp["solar"])[0, 1]
rho_ws_m = nat_m[["wind_gwh", "solar_gwh"]].corr().iloc[0, 1]
ax = axes[1]
vals = [ws["wind"].std(), ws["solar"].std(), ws["wind + solar"].std()]
bars = ax.bar(["Wind", "Solar", "Wind + solar"], vals, color=[cfg.COLOR["wind"], cfg.COLOR["solar"], "#555555"])
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.2f}", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Seasonal variation (std, TWh)"); ax.set_title("Do wind and solar compensate each other?")
viz.suptitle(fig, f"Wind and solar seasonal shapes are {'opposite' if rho_ws < -0.3 else 'not opposite'} (correlation {rho_ws:+.2f})", "Seasonal component of the national monthly production, by calendar month", y=1.04)
plt.tight_layout(); cards.save(fig, "F14_complementarity"); plt.show()

strength_tbl = pd.DataFrame({cfg.LABEL[s]: {"seasonal strength": comp[s + "_fs"], "trend strength": comp[s + "_ft"]} for s in cfg.RENEWABLE}).T
display(strength_tbl.round(2))
red = (1 - ws["wind + solar"].std() / (ws["wind"].std() + ws["solar"].std())) * 100
cards.add("F14", "Time", "Complementarity of wind and solar",
    "Are the seasonal patterns of the renewable sources complementary, and which sources are the most seasonal?",
    f"The correlation between the seasonal shapes of wind and solar is {rho_ws:+.2f} (monthly values: {rho_ws_m:+.2f}). Together their seasonal variation is {ws['wind + solar'].std():.2f} TWh, against {ws['wind'].std():.2f} for wind and {ws['solar'].std():.2f} for solar: the combination removes {red:.0f} % of the sum of the two variations. The most seasonal source is {strength_tbl['seasonal strength'].idxmax()} (strength {strength_tbl['seasonal strength'].max():.2f}) and the least seasonal {strength_tbl['seasonal strength'].idxmin()} ({strength_tbl['seasonal strength'].min():.2f}).",
    ("Wind and solar partly compensate each other over the year, which is a good argument for developing both. " if rho_ws < -0.3 else "Wind and solar do not clearly compensate each other in this data, so I should not claim it. ") + "I only see the monthly national total here: it says nothing about the hour of the day, nor about storage or consumption, which are outside the dataset.",
    "A positive and visual message (the two sources work together) that fits the audience, with a clear limit that I must state.")

## 5 · Regional gaps (Q7)
Are the regions getting closer or further apart? I measure the dispersion of the 13 regional renewable shares year by year: standard deviation, range (max - min) and coefficient of variation. A falling standard deviation is called sigma-convergence; I also check the link between the starting level and the gain (done in F09).

In [ ]:
g = ann.groupby("year")["renewable_share_pct"]
gap = pd.DataFrame({"std": g.std(), "range": g.max() - g.min(), "iqr": g.quantile(0.75) - g.quantile(0.25), "mean_regions": g.mean(), "cv_pct": g.std() / g.mean() * 100})
gn = ann.groupby("year")["new_renewables_share_pct"]
gap["std_new"] = gn.std()
grp = ann.assign(has_nuclear=ann["region_code"].map(HAS_NUC)).groupby(["year", "has_nuclear"])["renewable_share_pct"].mean().unstack()
gap["between_groups"] = grp[False] - grp[True]
gap["std_within_nuclear"] = ann[ann.region_code.map(HAS_NUC)].groupby("year")["renewable_share_pct"].std()
gap["std_within_no_nuclear"] = ann[~ann.region_code.map(HAS_NUC)].groupby("year")["renewable_share_pct"].std()
display(gap.round(2))

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8))
ax = axes[0]
ax.plot(gap.index, gap["std"], color="#0072B2", lw=2.4, marker="o", label="Std of the 13 regional shares")
ax.plot(gap.index, gap["range"], color="#D55E00", lw=2.2, marker="s", label="Range (max - min)")
ax.plot(gap.index, gap["std_new"], color="#009E73", lw=2.2, marker="^", label="Std, wind + solar + bioenergy share")
ax.set_ylabel("Percentage points"); ax.set_title("Dispersion between regions"); ax.legend(fontsize=8)
ax = axes[1]
for k in (False, True):
    ax.plot(grp.index, grp[k], color=viz.GROUP_COLOR[k], lw=2.4, marker="o", label=f"Mean, regions {viz.GROUP_LABEL[k]}")
    sub = ann[ann.region_code.map(HAS_NUC) == k].groupby("year")["renewable_share_pct"]
    ax.fill_between(grp.index, sub.min(), sub.max(), color=viz.GROUP_COLOR[k], alpha=0.15)
ax.set_ylabel("Renewable share (%)"); ax.set_title("Two groups: with and without nuclear"); ax.legend(fontsize=8)
sd0, sd1 = gap.loc[Y0, "std"], gap.loc[Y1, "std"]
slope_sd = stats.linregress(gap.index, gap["std"]).slope
viz.suptitle(fig, f"Regional gaps {'widen' if sd1 > sd0 else 'narrow'}: std {sd0:.1f} to {sd1:.1f} points", "Annual renewable share of the 13 regions (unweighted)", y=1.04)
plt.tight_layout(); cards.save(fig, "F15_regional_gaps"); plt.show()

cards.add("F15", "Time", "Are regional gaps increasing or decreasing? (Q7)",
    "Are the regions converging toward the same renewable share, or moving apart?",
    f"The standard deviation of the 13 regional shares {viz.direction(sd1-sd0, 'rose', 'fell')} from {sd0:.1f} pts in {Y0} to {sd1:.1f} pts in {Y1} (trend {slope_sd:+.2f} pts per year). The range {viz.direction(gap.loc[Y1,'range']-gap.loc[Y0,'range'], 'rose', 'fell', 'stayed about the same', tol=1)} from {gap.loc[Y0,'range']:.0f} to {gap.loc[Y1,'range']:.0f} pts, and the dispersion of the wind + solar + bioenergy share goes from {gap.loc[Y0,'std_new']:.1f} to {gap.loc[Y1,'std_new']:.1f} pts. The gap between the two groups (without minus with nuclear) is {gap.loc[Y0,'between_groups']:.1f} pts in {Y0} and {gap.loc[Y1,'between_groups']:.1f} pts in {Y1}. Inside the groups, the standard deviation is {gap.loc[Y1,'std_within_no_nuclear']:.1f} (without nuclear) and {gap.loc[Y1,'std_within_nuclear']:.1f} (with nuclear) in {Y1}.",
    ("The regions are moving apart: the ones that were ahead progress faster. " if sd1 > sd0 else "The regions are getting closer: the ones that were behind progress faster. ") + f"Most of the gap is {'between' if abs(gap.loc[Y1,'between_groups']) > gap.loc[Y1,'std'] else 'inside'} the two groups, so the nuclear / no nuclear divide {'explains a large part of' if abs(gap.loc[Y1,'between_groups']) > gap.loc[Y1,'std'] else 'is not the only source of'} the dispersion. The answer depends on the indicator (all renewables or only new sources), and I will say it." + f" For the new sources (wind + solar + bioenergy) the dispersion {viz.direction(gap.loc[Y1,'std_new']-gap.loc[Y0,'std_new'], 'widens', 'narrows')} ({gap.loc[Y0,'std_new']:.1f} to {gap.loc[Y1,'std_new']:.1f} pts).",
    "Direct answer to Q7 and a natural 'so what' for the audience: if gaps grow, regional policy matters; if they shrink, the transition is spreading.")

## 6 · Transition indicators: correlations, distributions and growth
Fifth level of the brief. I use the region-year table (13 regions x 12 years = 156 observations) with the shares, the diversity index and the two scores built in notebook 02. I use Spearman correlation because the relations are not necessarily linear and some distributions are skewed.

In [ ]:
cols = {"renewable_share_pct": "Renewable share", "new_renewables_share_pct": "Wind+solar+bio share", "hydro_share_pct": "Hydro share",
        "wind_share_pct": "Wind share", "solar_share_pct": "Solar share", "bioenergy_share_pct": "Bioenergy share",
        "nuclear_share_pct": "Nuclear share", "thermal_share_pct": "Thermal share", "mix_diversity_12m": "Mix diversity",
        "energy_transition_score": "Transition score", "renewable_momentum_score": "Momentum score", "total_gwh": "Total production"}
cm_ = ann[list(cols)].rename(columns=cols).corr(method="spearman")
cm_s = nat_m[gcols].rename(columns={cfg.GWH[s]: cfg.LABEL[s] for s in src}).corr(method="spearman")
fig, axes = plt.subplots(1, 2, figsize=(15, 6.2), gridspec_kw={"width_ratios": [1.6, 1]})
for ax, m_, ttl in [(axes[0], cm_, "Regions x years (156 observations)"), (axes[1], cm_s, "National monthly production by source")]:
    mask = np.triu(np.ones_like(m_, dtype=bool), k=1)
    im = ax.imshow(np.where(mask, np.nan, m_.values), cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(m_))); ax.set_xticklabels(m_.columns, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(m_))); ax.set_yticklabels(m_.index, fontsize=8)
    for i in range(len(m_)):
        for j in range(i + 1):
            v = m_.values[i, j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7, color="white" if abs(v) > 0.6 else viz.INK)
    ax.grid(False); ax.set_title(ttl); [ax.spines[s].set_visible(False) for s in ax.spines]
fig.colorbar(im, ax=axes, fraction=0.02, pad=0.02).outline.set_visible(False)
viz.suptitle(fig, "Which indicators move together?", "Spearman correlation", y=1.0)
cards.save(fig, "F16_correlations"); plt.show()

mech = [{"Renewable share", "Hydro share"}, {"Renewable share", "Wind share"}, {"Renewable share", "Solar share"}, {"Renewable share", "Bioenergy share"},
        {"Renewable share", "Wind+solar+bio share"}, {"Wind+solar+bio share", "Wind share"}, {"Wind+solar+bio share", "Solar share"},
        {"Wind+solar+bio share", "Bioenergy share"}, {"Renewable share", "Nuclear share"}, {"Renewable share", "Transition score"},
        {"Wind+solar+bio share", "Transition score"}, {"Thermal share", "Transition score"}, {"Transition score", "Mix diversity"}, {"Renewable share", "Thermal share"}]
pairs = [(a_, b_, cm_.loc[a_, b_]) for i, a_ in enumerate(cm_.index) for b_ in cm_.index[:i] if {a_, b_} not in mech]
pairs = sorted(pairs, key=lambda p: -abs(p[2]))
print("strongest correlations once the mechanical ones are removed (shares are built from each other):")
for a_, b_, v in pairs[:8]:
    print(f"  {a_:22s} - {b_:22s} {v:+.2f}")
r_hw = cm_s.loc["Hydro", "Wind"]; r_sw = cm_s.loc["Solar", "Wind"]; r_hs = cm_s.loc["Hydro", "Solar"]; r_th = cm_s.loc["Thermal", "Solar"]
cards.add("F16", "Indicators", "Correlation heatmaps",
    "Which indicators of the transition move together, and are some of them redundant?",
    "Leaving aside the pairs that are linked by construction (the shares add up to 100), the strongest links are: " + "; ".join(f"{a_} / {b_} {v:+.2f}" for a_, b_, v in pairs[:4]) + f". On the national monthly series, solar and wind are correlated at {r_sw:+.2f}, hydro and wind at {r_hw:+.2f}, hydro and solar at {r_hs:+.2f}, and thermal and solar at {r_th:+.2f}. The transition score and the momentum score are correlated at {cm_.loc['Transition score', 'Momentum score']:+.2f}.",
    ("The transition score and the momentum score are not strongly linked, so they measure two different things (level and speed), which is what I wanted. " if abs(cm_.loc['Transition score', 'Momentum score']) < 0.6 else "The two scores are quite correlated, so one of them could be enough for a simple view. ") + "Correlation here describes 156 region-year values that are not independent (same region every year), so I read it as a pattern and not as a test." + (f" The negative link between thermal and solar ({r_th:+.2f}) is the seasonal pattern: thermal is used when the sun is low." if r_th < -0.3 else ""),
    "Supports the choice of indicators for the dashboard (what is redundant, what is complementary) and gives numbers for the wind / solar / hydro complementarity.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), gridspec_kw={"width_ratios": [1.3, 1, 1]})
ax = axes[0]
yrs = sorted(ann.year.unique())
bp = ax.boxplot([ann.loc[ann.year == y, "renewable_share_pct"] for y in yrs], patch_artist=True, showfliers=False, medianprops=dict(color=viz.INK))
ax.set_xticks(range(1, len(yrs) + 1)); ax.set_xticklabels(yrs)
for p in bp["boxes"]:
    p.set_facecolor("#BBD7EC")
for i, y in enumerate(yrs, start=1):
    sub = ann[ann.year == y]
    ax.scatter(np.random.default_rng(1).normal(i, 0.06, len(sub)), sub["renewable_share_pct"], c=[viz.GROUP_COLOR[HAS_NUC[c]] for c in sub["region_code"]], s=14, zorder=3)
ax.set_ylabel("Renewable share (%)"); ax.set_title("Distribution across regions, by year"); ax.tick_params(axis="x", rotation=60)
ax = axes[1]
e_ = feat[feat.year <= Y0 + 2]["renewable_share_pct"].dropna(); l_ = feat[feat.year >= Y1 - 2]["renewable_share_pct"].dropna()
bins = np.linspace(0, 100, 26)
ax.hist(e_, bins=bins, alpha=0.65, color="#56B4E9", label=f"{Y0}-{Y0+2}", density=True)
ax.hist(l_, bins=bins, alpha=0.65, color="#D55E00", label=f"{Y1-2}-{Y1}", density=True)
ax.set_xlabel("Monthly renewable share of a region (%)"); ax.set_ylabel("Density"); ax.set_title("Distribution of region-months"); ax.legend()
ax = axes[2]
gc = {"renewable_growth_pct": "Renewables", "new_renewables_growth_pct": "Wind+solar+bio", "solar_growth_pct": "Solar", "wind_growth_pct": "Wind"}
gdata = [feat[c].dropna().clip(-100, 300) for c in gc]
bp = ax.boxplot(gdata, patch_artist=True, showfliers=False, medianprops=dict(color=viz.INK))
ax.set_xticks(range(1, len(gc) + 1)); ax.set_xticklabels(list(gc.values()))
for p, c in zip(bp["boxes"], ["#0072B2", "#E69F00", cfg.COLOR["solar"], cfg.COLOR["wind"]]):
    p.set_facecolor(c); p.set_alpha(0.6)
ax.axhline(0, color="#999", lw=0.8); ax.set_ylabel("Year-on-year growth (%), clipped at -100 / 300"); ax.set_title("Growth of region-months")
skew_e, skew_l = stats.skew(e_), stats.skew(l_)
viz.suptitle(fig, f"The distribution of renewable shares {viz.direction(l_.median()-e_.median(), 'moved up', 'moved down', 'did not move')} (median {e_.median():.0f} % to {l_.median():.0f} %)", "Region-years and region-months", y=1.04)
plt.tight_layout(); cards.save(fig, "F17_distributions"); plt.show()

gmed = {v: float(feat[k].median()) for k, v in gc.items()}
gp_ = {v: float((feat[k].dropna() > 0).mean() * 100) for k, v in gc.items()}
cards.add("F17", "Indicators", "Distribution analysis and boxplots",
    "How are the renewable shares distributed across regions and months, and how skewed and volatile is the growth?",
    f"The median regional renewable share of a month {viz.direction(l_.median()-e_.median(), 'rose', 'fell')} from {e_.median():.0f} % in {Y0}-{Y0+2} to {l_.median():.0f} % in {Y1-2}-{Y1}; the interquartile range goes from {e_.quantile(.75)-e_.quantile(.25):.0f} to {l_.quantile(.75)-l_.quantile(.25):.0f} pts. The skewness goes from {skew_e:+.2f} to {skew_l:+.2f}. The median year-on-year growth is " + ", ".join(f"{k} {v:+.1f} %" for k, v in gmed.items()) + "; the share of region-months with positive growth is " + ", ".join(f"{k} {v:.0f} %" for k, v in gp_.items()) + ".",
    ("The whole distribution moves, not only the top regions. " if l_.quantile(.25) > e_.quantile(.25) else "The upper part moves more than the lower part, so the progress is not shared by all regions. ") + "Solar and wind growth are very dispersed (the boxes are wide even after clipping), because a small base makes the percentages explode; for comparisons I will prefer points of share and TWh.",
    "Shows that I can trust medians and quartiles but not means of growth rates. It is a methodological point I must respect in the dashboard (no average of percentages).")

In [ ]:
add = pd.DataFrame({cfg.LABEL[s]: (a1[cfg.GWH[s]] - a0[cfg.GWH[s]]) / 1000 for s in cfg.RENEWABLE}).loc[reg.index]
add["new_total"] = add[["Wind", "Solar", "Bioenergy"]].sum(axis=1)
add = add.sort_values("new_total")
nr = ann.pivot(index="region", columns="year", values="new_renewables_gwh")
nr_g = (nr.pct_change(axis=1) * 100).drop(columns=Y0)
nr_g = nr_g.loc[[NAME[c] for c in reg.sort_values("new_gain", ascending=False).index]]
fig, axes = plt.subplots(1, 2, figsize=(14, 5.4), gridspec_kw={"width_ratios": [1, 1.25]})
ax = axes[0]
left_pos = np.zeros(len(add)); left_neg = np.zeros(len(add))
for s in ["Hydro", "Bioenergy", "Wind", "Solar"]:
    v = add[s].to_numpy(); lp = np.where(v >= 0, left_pos, left_neg)
    ax.barh([NAME[c] for c in add.index], v, left=lp, color=cfg.COLOR[[k for k, vv in cfg.LABEL.items() if vv == s][0]], label=s, height=0.75)
    left_pos += np.where(v >= 0, v, 0); left_neg += np.where(v < 0, v, 0)
ax.axvline(0, color="#777", lw=0.9); ax.set_xlabel(f"TWh added, {Y1} minus {Y0}"); ax.set_title("Renewable volume added"); ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.1))
ax = axes[1]
im = ax.imshow(nr_g.values, aspect="auto", cmap="RdBu_r", vmin=-60, vmax=60)
ax.set_xticks(range(nr_g.shape[1])); ax.set_xticklabels(nr_g.columns, fontsize=8); ax.set_yticks(range(nr_g.shape[0])); ax.set_yticklabels(nr_g.index, fontsize=8)
for i in range(nr_g.shape[0]):
    for j in range(nr_g.shape[1]):
        ax.text(j, i, f"{nr_g.values[i, j]:.0f}", ha="center", va="center", fontsize=7, color="white" if abs(nr_g.values[i, j]) > 40 else viz.INK)
ax.grid(False); [ax.spines[s].set_visible(False) for s in ax.spines]
cb = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02); cb.outline.set_visible(False); cb.set_label("% vs previous year (colour clipped at +-60)")
ax.set_title("Growth of wind + solar + bioenergy, year on year")
big_r = add["new_total"].idxmax()
viz.suptitle(fig, f"{NAME[big_r]} adds the most wind, solar and bioenergy ({add.loc[big_r, 'new_total']:.1f} TWh)", "Absolute volumes (left) and year-on-year growth (right), by region", y=1.03)
plt.tight_layout(); cards.save(fig, "F18_growth_by_region"); plt.show()

conc3 = add["new_total"].sort_values(ascending=False).head(3).sum() / add["new_total"].sum() * 100
neg_h = int((add["Hydro"] < 0).sum())
cards.add("F18", "Indicators", "Growth analysis by region",
    "Where do the new renewable volumes come from, and is the growth regular from one year to the next?",
    f"The three regions that add the most wind, solar and bioenergy are {', '.join(NAME[c] for c in add['new_total'].sort_values(ascending=False).head(3).index)}; together they represent {conc3:.0f} % of the volume added in France. The total added by all regions is {add['new_total'].sum():.1f} TWh. Hydro {viz.direction(add['Hydro'].sum(), 'adds', 'loses')} {abs(add['Hydro'].sum()):.1f} TWh in total and falls in {neg_h} region(s). In the heatmap, {int((nr_g.values < 0).sum())} region-years out of {nr_g.size} show a fall of the new renewables year on year.",
    ("The growth is concentrated in a few regions: the national story is driven by them. " if conc3 > 55 else "The growth is spread over many regions, so the national trend is not the work of just two or three of them. ") + "Falls in the heatmap are rare and mostly small, which suggests that the capacity keeps growing even when the weather changes; the exact reasons are not in the data.",
    "Connects the national trend to places: who really builds the transition. Good material for a ranking page and for a 'region in focus' view.")

## 7 · What surprised me (Q8)
A surprising finding is one that contradicts a simple expectation or a simple reading of the numbers. I tested the following expectations and kept the ones where the data disagrees:
1. "The region with the highest renewable share is the leader of the transition."
2. "The national renewable share is the average of the regional shares."
3. "A higher renewable share means more renewable electricity."
4. "The transition is steady."
5. "Hydro is the base of the renewables, so it keeps the same weight."

In [ ]:
wm = ann.groupby("year").apply(lambda d: pd.Series({"mean_regions": d["renewable_share_pct"].mean(), "weighted_national": d["renewable_gwh"].sum() / d["total_gwh"].sum() * 100}))
gap_mw = wm["mean_regions"] - wm["weighted_national"]
cf_series = natv["renewable_gwh"] / tg0 * 100
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))
ax = axes[0]
ax.plot(wm.index, wm["mean_regions"], color="#CC79A7", lw=2.4, marker="o", label="Mean of the 13 regional shares")
ax.plot(wm.index, wm["weighted_national"], color="#0072B2", lw=2.4, marker="o", label="National share (weighted by production)")
ax.fill_between(wm.index, wm["weighted_national"], wm["mean_regions"], color="#CC79A7", alpha=0.12)
ax.set_ylabel("Renewable share (%)"); ax.set_title("Two ways to compute 'the' share"); ax.legend(fontsize=8, loc="upper left")
ax = axes[1]
ax.plot(natv.index, natv["renewable_share_pct"], color="#0072B2", lw=2.4, marker="o", label="Actual national share")
ax.plot(natv.index, cf_series, color="#7F7F7F", lw=2.2, ls="--", marker="o", ms=4, label=f"If total production had stayed at its {Y0} level")
ax.set_ylabel("Renewable share (%)"); ax.set_title("How much of the change is the denominator?"); ax.legend(fontsize=8, loc="upper left")
viz.suptitle(fig, f"The mean of the regions ({wm['mean_regions'].iloc[-1]:.0f} %) is not the national share ({wm['weighted_national'].iloc[-1]:.0f} %)", "Two traps when reading a renewable share", y=1.04)
plt.tight_layout(); cards.save(fig, "F19_surprises_measures"); plt.show()

cards.add("F19", "Surprises", "Two measures that look the same and are not (Q8)",
    "Are the numbers that I will show on the dashboard comparable, and can a share rise without more renewable production?",
    f"In {Y1} the mean of the 13 regional shares is {wm['mean_regions'].iloc[-1]:.1f} % while the national share weighted by production is {wm['weighted_national'].iloc[-1]:.1f} % (a difference of {gap_mw.iloc[-1]:+.1f} pts; {gap_mw.iloc[0]:+.1f} pts in {Y0}). Total production {viz.direction(tg1-tg0, 'rose', 'fell')} by {abs(tg1/tg0-1)*100:.1f} % from {Y0} to {Y1}; with the {Y0} total, the {Y1} renewable output would give {cf_series.iloc[-1]:.1f} % instead of {sh1:.1f} %, so {sh1-cf_series.iloc[-1]:+.1f} pts of the change come from the denominator.",
    f"The big nuclear regions weigh a lot in the national total but count only 1/13 in the simple mean, which is why the two measures differ. A dashboard that mixes them would show numbers that do not match from one page to another. " + ("A share can rise because total production falls, so I will pair every share with a volume in TWh." if abs(sh1 - cf_series.iloc[-1]) > 0.1 * abs(sh1 - sh0) else "Here the denominator effect is small."),
    "A real surprise for the audience and a strong 'be careful' message. It also protects the credibility of the story: I will define the measures once and use them consistently.")

## 8 · Ranking of all the findings
I list every finding with the numbers that support it, and I score it on four criteria from 1 to 5:
- **Impact**: does it change the answer to the research question?
- **Evidence**: how solid is the number (size of the effect, consistency, risk of a weather or base effect)?
- **Surprise**: would the audience expect it?
- **Storyability**: can it be shown in one clear chart?

`score = 0.35 × impact + 0.25 × evidence + 0.20 × surprise + 0.20 × storyability`. A finding is **Major** from 4.2, **Supporting** from 3.3 and **Interesting** below. The scores are my judgement, written in the code so I can change them; the evidence column is computed.

In [ ]:
n_ext = int(feat["flag_extreme_value"].sum())
hydro_flat = abs(add_by["hydro"]) < 0.25 * abs(add_tot)
nuc_divide = abs(a1.loc[no_nuc_regs, "renewable_share_pct"].mean() - a1.loc[nuc_regs, "renewable_share_pct"].mean()) > 15
F = [
 ("Wind, solar and bioenergy drive the growth of renewables" + ("; hydro is flat" if hydro_flat else "; hydro also moves"), "Sources",
  f"{sh_new_add:.0f} % of the extra renewable TWh is wind + solar + bioenergy; hydro {add_by['hydro']/1000:+.1f} TWh", "F02, F04, F05", 5, 5, 3, 5, sh_new_add > 60),
 ("The renewable share rose, but not in a straight line", "France",
  f"{sh0:.1f} % to {sh1:.1f} % ({gain_total:+.1f} pts); worst yearly step {d_sh.min():+.1f} pts in {d_sh.idxmin()}", "F03", 5, 5, 2, 4, True),
 ("The level of the share " + ("mostly separates" if nuc_divide else "does not clearly separate") + " regions with and without nuclear", "Regions",
  f"mean {a1.loc[no_nuc_regs, 'renewable_share_pct'].mean():.0f} % without nuclear vs {a1.loc[nuc_regs, 'renewable_share_pct'].mean():.0f} % with; {int((top.head(5)['has_nuclear']==False).sum())}/5 highest shares are regions without nuclear", "F06, F07", 5, 4, 4, 5, nuc_divide),
 ("Level and progress give " + ("different" if overlap <= 2 else "partly different" if overlap == 3 else "similar") + " leaders: four profiles of regions", "Regions",
  f"only {overlap} region(s) in both top 5 (level / gain); profiles: " + "; ".join(f"{k}: {v}" for k, v in prof.items()), "F08, F09", 5, 4, 4, 5, overlap < 4),
 ("Fastest source is " + ("not " if fast != big else "") + "the biggest contributor", "Sources",
  f"{fast} {fastest_row['cagr_pct']:.1f} %/yr vs {big} {biggest_row['added_twh']:+.1f} TWh added", "F04", 4, 5, 3, 4, fast != big),
 ("Regional gaps " + ("widen" if sd1 > sd0 else "narrow"), "Time / regions",
  f"std of regional shares {sd0:.1f} to {sd1:.1f} pts; range {gap.loc[Y0,'range']:.0f} to {gap.loc[Y1,'range']:.0f} pts; between-group gap {gap.loc[Y0,'between_groups']:.1f} to {gap.loc[Y1,'between_groups']:.1f} pts; std of the new-source share {gap.loc[Y0,'std_new']:.1f} to {gap.loc[Y1,'std_new']:.1f} pts", "F15", 4, 4, 3, 4, True),
 ("Wind and solar are seasonal complements" if rho_ws < -0.3 else "Wind and solar are not clear seasonal complements", "Time",
  f"correlation of seasonal shapes {rho_ws:+.2f}; combined seasonal variation {red:.0f} % lower than the sum of the two", "F14", 4, 3, 3, 4, rho_ws < -0.3),
 ("Seasonality " + ("dominates" if fs > ft else "is weaker than the trend in") + " the monthly series", "Time",
  f"seasonal strength {fs:.2f} vs trend strength {ft:.2f}; peak month {MONTHS[pk-1]}, weakest {MONTHS[tr-1]}", "F12, F13", 3, 5, 2, 3, fs > ft),
 ("Mean of the regions and national share are different numbers", "Method",
  f"{wm['mean_regions'].iloc[-1]:.1f} % vs {wm['weighted_national'].iloc[-1]:.1f} % in {Y1}", "F19", 3, 5, 4, 3, abs(wm["mean_regions"].iloc[-1] - wm["weighted_national"].iloc[-1]) > 2),
 ("Part of the share gain comes from the denominator" if abs(sh1 - cf_series.iloc[-1]) > 0.15 * abs(gain_total) else "The denominator effect on the share is small", "Method",
  f"{sh1-cf_series.iloc[-1]:+.1f} pts of the {gain_total:+.1f} pts if total production had stayed at the {Y0} level", "F03, F19", 3, 4, 4, 3, abs(sh1 - cf_series.iloc[-1]) > 0.15 * abs(gain_total)),
 ("A few regions add most of the new renewable volume" if conc3 > 55 else "The new renewable volume is spread over many regions", "Regions",
  f"top 3 regions = {conc3:.0f} % of the wind + solar + bioenergy TWh added", "F18", 4, 4, 3, 4, True),
 ("Renewables replaced mostly nuclear" if nuc_main else "Renewables replaced mostly thermal" if d["Thermal"] < 0 else "Renewables grew next to the other sources", "France",
  f"nuclear {d['Nuclear']:+.1f} pts ({add_nuc:+.1f} TWh) vs thermal {d['Thermal']:+.1f} pts ({add_the:+.1f} TWh); renewables {pos_ren:+.1f} pts ({add_tot/1000:+.1f} TWh)", "F01, F05", 5, 5, 5, 4, nuc_main),
 ("Regional ranking is " + ("stable" if rho_rank.statistic > 0.8 else "not very stable"), "Regions",
  f"Spearman rank correlation of the share between {Y0} and {Y1}: {rho_rank.statistic:.2f}", "F09, F11", 3, 4, 3, 3, True),
 ("A few extreme solar months exist", "Data",
  f"{n_ext} region-months flagged as extreme (all in solar in notebook 02)", "F17", 2, 3, 2, 1, True),
]
ins = pd.DataFrame(F, columns=["finding", "level", "evidence", "figures", "impact", "evidence_strength", "surprise", "storyability", "as_expected"])
# a finding that does not hold in the data (the expectation is false) is kept but loses most of its weight
ins.loc[~ins["as_expected"], ["impact", "surprise", "storyability"]] = ins.loc[~ins["as_expected"], ["impact", "surprise", "storyability"]].sub(2).clip(lower=1)
ins["score"] = (0.35 * ins["impact"] + 0.25 * ins["evidence_strength"] + 0.20 * ins["surprise"] + 0.20 * ins["storyability"]).round(2)
ins["class"] = np.where(ins["score"] >= 4.2, "Major Insight", np.where(ins["score"] >= 3.3, "Supporting Insight", "Interesting Observation"))
ins = ins.sort_values("score", ascending=False).reset_index(drop=True)
ins.index = ins.index + 1; ins.index.name = "rank"
display(ins[["finding", "class", "score", "evidence", "figures"]])
print(ins["class"].value_counts().to_string())

**What I conclude.** The strongest findings are about what changes in the mix (wind and solar grow, and nuclear gives way more than thermal), about regions (level versus progress, and a gap between regions with and without nuclear that does not close) and about method (mean versus weighted share). The seasonal and gap findings support them. The findings of the "Interesting" class are useful as details but not as the main message.

## 9 · Storytelling candidates
From the ranked findings I can see three possible stories. I score them on five criteria (1 to 5): clarity, evidence, novelty for the audience, relevance to the research question, and how much the audience can do with it.

In [ ]:
stories = pd.DataFrame([
  ("S1", "Wind and solar, not hydro, are rewriting the French electricity mix",
   "National and sources", "F01, F02, F03, F04, F05", 5, 5, 3, 4, 3),
  ("S2", "Two Frances: the regions that are ahead and the regions that are moving",
   "Regions", "F06, F07, F08, F09, F10, F15", 4, 4, 5, 5, 4),
  ("S3", "Sun in summer, wind in winter: the transition is a question of seasons",
   "Time", "F12, F13, F14", 4, 3, 3, 3, 3),
], columns=["id", "title", "scale", "figures", "clarity", "evidence", "novelty", "relevance", "actionability"])
stories["total"] = stories[["clarity", "evidence", "novelty", "relevance", "actionability"]].sum(axis=1)
stories = stories.sort_values("total", ascending=False).reset_index(drop=True)
display(stories)
chosen = stories.iloc[0]
print(f"story with the highest score: {chosen['id']} - {chosen['title']} ({chosen['total']}/25)")

**My choice.** I keep **S2 as the main story** because it answers the research question directly (regions, progress) and it has the most surprising result: the level ranking and the progress ranking do not designate the same regions. I use **S1 as the opening** (one chart of the national picture and the sources, so that the audience understands what "progress" means) and **S3 as a short closing point** (what the share does not show: the season). The narrative arc is written in `storytelling_candidates.md`.

## 10 · Written outputs
The four reports are generated from the numbers above, so they always match the figures.

In [ ]:
today = pd.Timestamp.today().date()
sev = lambda df: df.to_markdown(index=False) if hasattr(df, "to_markdown") else df.to_string(index=False)
try:
    import tabulate  # noqa: F401
    mdtab = lambda df, index=False: df.to_markdown(index=index)
except Exception:
    from utils.prep import to_markdown_table as _t
    mdtab = lambda df, index=False: _t(df.reset_index() if index else df)

ins_md = ins.reset_index()[["rank", "finding", "class", "score", "evidence", "figures"]]

eda = [f"""# EDA report

Project: Renewable energy transition in France - regional dynamics
Author: Soumahoro Souleymane
Produced by `notebooks/03_eda_storytelling.ipynb` on {today}

## 1. Scope

Research question: *how are French regions progressing toward a renewable electricity future?*
Data: {feat.shape[0]:,} region-months ({feat.region_code.nunique()} metropolitan regions, {feat.date.min():%Y-%m} to {feat.date.max():%Y-%m}), prepared in notebook 02. National values are sums of the 13 regions, so national shares are weighted by production. Shares of a region use production, not consumption.

## 2. Method notes

- Level = renewable share of the regional production in the last year; progress = change of this share since {Y0} (points).
- Rates of growth are given with the volume added (TWh), because a rate on a small base is misleading.
- Comparisons of the 3 first and the 3 last years are used to check that one year (weather) does not drive a result.
- Seasonal decomposition: additive, centred 12-month moving average (own implementation in `utils/viz.py`).
- Correlations are Spearman; 156 region-years are not independent observations, so they describe patterns and do not test hypotheses.
- With 13 regions, every statistic about regions is descriptive.

## 3. Answers to the eight questions

| Q | Short answer |
|---|---|
| Q1 | Renewable production {viz.direction(chg, 'rose', 'fell')} by {abs(chg):.0f} % ({ren0/1000:,.0f} to {ren1/1000:,.0f} TWh); {sh_new_add:.0f} % of the increase is wind, solar and bioenergy. |
| Q2 | By level: {', '.join(top.head(3)['region'])}. By progress: {', '.join(gp.head(3)['region'])}. |
| Q3 | By level: {', '.join(bot.head(3)['region'])}. Lagging profile (low level and low progress): {prof.get('Lagging', 'none')}. |
| Q4 | Fastest rate: {fast} ({fastest_row['cagr_pct']:.1f} % per year). Largest volume: {big} ({biggest_row['added_twh']:+.1f} TWh). |
| Q5 | The weighted national share {viz.direction(gain_total)} from {sh0:.1f} % to {sh1:.1f} % ({gain_total:+.1f} pts). |
| Q6 | Renewable share peaks in {MONTHS[pk-1]} and is lowest in {MONTHS[tr-1]}; seasonal strength {fs:.2f}; wind / solar seasonal correlation {rho_ws:+.2f}. |
| Q7 | The dispersion between regions {viz.direction(sd1-sd0, 'increases', 'decreases')} (std {sd0:.1f} to {sd1:.1f} pts). |
| Q8 | See section 5 (ranked findings): level versus progress, nuclear divide, mean versus weighted share, denominator effect. |

## 4. Figures
"""]
for sec in dict.fromkeys(c["section"] for c in cards.items):
    eda.append(f"\n### {sec}\n")
    for c in [c for c in cards.items if c["section"] == sec]:
        eda.append(viz.Cards.as_markdown(c, heading=4, with_image=True))
eda.append(f"""
## 5. Ranked findings

{mdtab(ins_md)}

## 6. Limits

- Production only: no consumption, imports/exports, capacity or prices.
- Net production: 34 negative values were set to 0 for the shares (0.0001 % of the energy, see notebook 02).
- Weather (rain, wind, sun) moves the yearly values; I use multi-year comparisons when possible.
- The classification of nuclear regions comes from the data (empty nuclear column), not from an external list.
- The causes of changes (policy, installations) are not in the data; I only describe.
""")
(REPORTS / "eda_report.md").write_text("\n".join(eda), encoding="utf-8")
print("eda_report.md:", len("\n".join(eda).splitlines()), "lines")

In [ ]:
major = ins[ins["class"] == "Major Insight"]
sup = ins[ins["class"] == "Supporting Insight"]
inte = ins[ins["class"] == "Interesting Observation"]
def block(df, title):
    out = [f"## {title}\n"]
    for r in df.itertuples():
        out.append(f"### {r.Index}. {r.finding}\n\n- Evidence: {r.evidence}\n- Figures: {r.figures}\n- Score: {r.score} (impact {r.impact}, evidence {r.evidence_strength}, surprise {r.surprise}, storyability {r.storyability})\n")
    return "\n".join(out) if len(df) else f"## {title}\n\nNone.\n"

key = f"""# Key insights

Project: Renewable energy transition in France - regional dynamics
Author: Soumahoro Souleymane, produced on {today}

Scoring: 0.35 x impact + 0.25 x evidence + 0.20 x surprise + 0.20 x storyability, each from 1 to 5. Major >= 4.2, Supporting >= 3.3.

{block(major, 'Major insights')}
{block(sup, 'Supporting insights')}
{block(inte, 'Interesting observations')}
## Caveats to keep in the story

- Shares are shares of production, not of consumption.
- The mean of the regional shares ({wm['mean_regions'].iloc[-1]:.1f} % in {Y1}) is not the national share ({wm['weighted_national'].iloc[-1]:.1f} %).
- Level partly reflects the absence of nuclear; progress is the fairer comparison.
- 13 regions: correlations and quadrants are descriptive.
"""
(REPORTS / "key_insights.md").write_text(key, encoding="utf-8")

arc = lambda d: "\n".join(f"- **{k}**: {v}" for k, v in d.items())
s2 = {"Hook": f"{gp.iloc[0]['region']} gained {gp.iloc[0]['gain']:+.1f} points in {YEARS} years, while {bot.iloc[0]['region']} is still at {bot.iloc[0]['share1']:.0f} % ({bot.iloc[0]['gain']:+.1f} pts).",
      "Context": f"France went from {sh0:.1f} % to {sh1:.1f} % renewable ({gain_total:+.1f} pts); {sh_new_add:.0f} % of the extra electricity is wind, solar and bioenergy (F02, F03).",
      "Tension": f"The ranking by level and the ranking by progress are different (only {overlap} region(s) in both top 5). The level mostly separates regions with and without nuclear (F06, F07).",
      "Evidence": f"Four profiles of regions (F09): " + "; ".join(f"{k}: {v}" for k, v in prof.items()) + f". Gaps between regions {viz.direction(sd1-sd0, 'increase', 'decrease')} (F15).",
      "Resolution": "Judge a region by its progress and its new-source volume, and always show its starting point.",
      "Call to action": "Look at your region in the dashboard: where is it, how fast is it moving, and what does it still depend on?"}
s1 = {"Hook": f"In {YEARS} years, wind, solar and bioenergy added {add_new/1000:.1f} TWh; hydro added {add_by['hydro']/1000:+.1f} TWh and nuclear {add_nuc:+.1f} TWh.",
      "Context": "France has a large nuclear base and hydro that depends on rain.",
      "Tension": "The national share moves but not in a straight line, and part of the gain comes from the denominator (F03, F19).",
      "Evidence": f"F02, F04, F05: {fast} grows fastest and {big} adds the most.",
      "Resolution": "The transition is a story of new sources replacing thermal and part of nuclear in the mix.",
      "Call to action": "Follow the sources, not only the total."}
s3 = {"Hook": f"The renewable share is {share_m.max():.0f} % in {MONTHS[pk-1]} and {share_m.min():.0f} % in {MONTHS[tr-1]}.",
      "Context": "A yearly share hides the monthly rhythm.",
      "Tension": f"Wind and solar peak at different moments (correlation {rho_ws:+.2f}) but the season stays the strongest pattern ({fs:.2f}).",
      "Evidence": "F12, F13, F14.", "Resolution": "Diversity of sources smooths the year.", "Call to action": "Look at the seasons before judging a share."}
cand = f"""# Storytelling candidates

Project: Renewable energy transition in France - regional dynamics
Author: Soumahoro Souleymane, produced on {today}

## Scoring

{mdtab(stories[['id', 'title', 'figures', 'clarity', 'evidence', 'novelty', 'relevance', 'actionability', 'total']])}

## S2 - {stories.set_index('id').loc['S2', 'title']}

{arc(s2)}

Risks: 13 regions only; level depends on nuclear; the causes (policy, installations) are not in the data.

## S1 - {stories.set_index('id').loc['S1', 'title']}

{arc(s1)}

Risks: the national picture is the most expected one; the share depends on the denominator.

## S3 - {stories.set_index('id').loc['S3', 'title']}

{arc(s3)}

Risks: national monthly totals only, no hours, storage or consumption.

## Recommendation

Main story: **{chosen['id']} - {chosen['title']}** ({chosen['total']}/25). Use S1 as the opening (the national context in 2 figures) and S3 as a closing nuance. Order of the narrative: national picture, then sources, then regions (level vs progress), then gaps, then seasons.
"""
(REPORTS / "storytelling_candidates.md").write_text(cand, encoding="utf-8")
print("key_insights.md and storytelling_candidates.md written")

In [ ]:
dash = f"""# Dashboard recommendations

Project: Renewable energy transition in France - regional dynamics
Author: Soumahoro Souleymane, produced on {today}
The dashboard is not built yet. These are the choices that the analysis supports.

## 1. Structure (one message per page)

| Page | Message | Content (figure it comes from) |
|---|---|---|
| 1. Overview | France in {Y1} versus {Y0} | 6 KPI tiles (F01), headline of the story |
| 2. The national picture | Wind and solar drive the change | stacked production + share (F02, F03), source contribution (F05) |
| 3. Sources | Rate and volume tell different things | index, rate, volume (F04), mix by region (F06) |
| 4. Regions | Level is not progress | map with a level / progress switch (F10), dumbbell (F07), quadrant (F09), ranking table |
| 5. Gaps | Are regions converging? | dispersion + group means (F15), heatmap (F11) |
| 6. Seasons | When is the electricity renewable? | seasonal profile, decomposition, wind vs solar (F12-F14) |
| 7. Method | How to read the numbers | definitions, weighted vs mean (F19), data notes |

## 2. Controls

- Year range slider ({Y0}-{Y1}) and a switch "calendar year / rolling 12 months".
- Region multiselect and a "with / without nuclear" filter (default: all).
- Measure switch: share (%) or volume (TWh). Every share is displayed next to a volume.
- Source selector (hydro, wind, solar, bioenergy; nuclear and thermal for the mix).
- Score weights sliders (transition score), with a reset button; the components are stored in the dataset.

## 3. KPIs (definitions to print under each number)

- Renewable share = renewable GWh / total GWh of the selected regions (weighted).
- Wind + solar + bioenergy share, hydro part of the renewables.
- Progress = change of the renewable share since the first selected year (points).
- Momentum score and transition score (0-100, relative to the dataset, see feature dictionary).

## 4. Rules that come from the analysis

1. Never average percentages across regions: use sums of GWh ({wm['mean_regions'].iloc[-1]:.1f} % mean of regions vs {wm['weighted_national'].iloc[-1]:.1f} % weighted in {Y1}).
2. Show level and progress together; explain the nuclear / no nuclear starting point with the peer rank.
3. Give rate and volume together for growth (a small base gives huge rates).
4. Use 12-month windows for comparisons; monthly growth is noisy.
5. Titles are messages ("Renewable output rose X %"), numbers come from the data.
6. Show the data note: negative net values set to 0 in the shares (0.0001 % of the energy).

## 5. Design and performance

- Okabe-Ito palette (colour-blind safe), the same colour per source on every page.
- Read `feature_engineered_dataset.csv` once with `st.cache_data`; use `annual_region_summary.csv` and `national_annual.csv` for fast pages; geometry from `regions.geojson`.
- Each chart has a one-line takeaway and a caption with the source.
- Export: PNG of each chart and CSV of the table shown.

## 6. What not to build

- No forecast (the data has no drivers).
- No ranking of regions by GWh only (size effect).
- No claim about consumption, imports or prices.
"""
(REPORTS / "dashboard_recommendations.md").write_text(dash, encoding="utf-8")
print("dashboard_recommendations.md written")
print()
print("figures saved:", len(list(FIGS.glob('*.png'))), "| cards:", len(cards.items))

## 11 · Summary of the run
This block lists what I need to finalise the commentary.

In [ ]:
print("=" * 78); print("RUN SUMMARY - notebook 03"); print("=" * 78)
print(f"period {Y0}-{Y1} | regions {feat.region_code.nunique()} | figures {len(list(FIGS.glob('*.png')))} | cards {len(cards.items)}")
print(f"national renewable: {ren0/1000:,.0f} -> {ren1/1000:,.0f} TWh ({chg:+.0f} %) | share {sh0:.1f} -> {sh1:.1f} % | new renewables {nsh0:.1f} -> {nsh1:.1f} %")
print(f"3-year averages share: {sh_early:.1f} -> {sh_late:.1f} %  | denominator effect {sh1-cf_series.iloc[-1]:+.1f} pts | mean of regions {wm['mean_regions'].iloc[-1]:.1f} vs weighted {wm['weighted_national'].iloc[-1]:.1f}")
print(f"added TWh by source: " + ", ".join(f"{k} {v/1000:+.1f}" for k, v in add_by.items()) + f" | fastest {fast}, biggest {big}")
print(growth_src[["cagr_pct", "cagr_3y_avg_pct", "added_twh", "multiple"]].round(2).to_string())
print(f"level top3: {', '.join(top.head(3)['region'])} | progress top3: {', '.join(gp.head(3)['region'])} | bottom level: {', '.join(bot.head(3)['region'])}")
print(f"profiles: {prof}")
print(f"overlap top5 level/gain: {overlap} | spearman start vs gain {rho_lg.statistic:+.2f} (p={rho_lg.pvalue:.2f}) | rank stability {rho_rank.statistic:.2f}")
print(f"gap std {sd0:.2f} -> {sd1:.2f} | range {gap.loc[Y0,'range']:.1f} -> {gap.loc[Y1,'range']:.1f} | between groups {gap.loc[Y0,'between_groups']:.1f} -> {gap.loc[Y1,'between_groups']:.1f}")
print(f"seasonal: peak {MONTHS[pk-1]}, low {MONTHS[tr-1]} | strength seasonal {fs:.2f} trend {ft:.2f} | wind-solar corr {rho_ws:+.2f}")
print(f"top3 concentration of new renewables added: {conc3:.0f} %")
print()
print(ins[["finding", "class", "score"]].to_string())